In [0]:
USE CATALOG maven_catalog;

In [0]:
USE SCHEMA gold_schema;

In [0]:
CREATE OR REPLACE VIEW maven_catalog.gold_schema.secured_fact_sales AS
SELECT 
    f.*,
    dr.sales_region
FROM maven_catalog.gold_schema.fact_sales f
LEFT JOIN maven_catalog.gold_schema.dim_stores dr
    ON f.store_id = dr.store_id
WHERE 
    /* 1. Admin/Data Engineer groups bypass */
    is_account_group_member('admins')
    OR is_account_group_member('grp_data_engineers')
    /* 2. Regional Manager Filtering */
    OR (
        is_account_group_member('regional_manager')
        AND dr.sales_region IN (
            SELECT allowed_sales_region
            FROM maven_catalog.governance.region_manager_access
            WHERE user_email = current_user()
        )
    );

In [0]:
CREATE OR REPLACE VIEW maven_catalog.gold_schema.secured_fact_returns AS
SELECT 
    fr.*,
    dr.sales_region
FROM maven_catalog.gold_schema.fact_returns fr
LEFT JOIN maven_catalog.gold_schema.dim_stores dr
    ON fr.store_id = dr.store_id
WHERE 
    /* 1. Admin/Data Engineer groups bypass */
    is_account_group_member('admins')
    OR is_account_group_member('grp_data_engineers')
    /* 2. Regional Manager Filtering */
    OR dr.sales_region IN (
        SELECT allowed_sales_region
        FROM maven_catalog.governance.region_manager_access
        WHERE user_email = current_user()
    );


In [0]:
-- Run as any admin user
SELECT COUNT(*) AS total_rows
FROM maven_catalog.gold_schema.secured_fact_sales;


In [0]:
select current_user();